## Example 1: Basic HTML to Markdown conversion

In [ ]:
import markdownify

# Example 1: Basic HTML to Markdown conversion
print("markdownify HTML to Markdown conversion")

try:
    print("Example 1 - Basic HTML to Markdown conversion")
    
    # Sample HTML content with various elements
    html_content = """
    <h1>Main Title</h1>
    <h2>Section Header</h2>
    <p>This is a paragraph with <strong>bold text</strong> and <em>italic text</em>.</p>
    <p>Here's another paragraph with a <a href="https://example.com">link to example</a>.</p>
    <ul>
        <li>First list item</li>
        <li>Second list item with <code>inline code</code></li>
        <li>Third item</li>
    </ul>
    <blockquote>
        <p>This is a blockquote with some important information.</p>
    </blockquote>
    <pre><code>def hello_world():
    print("Hello, World!")</code></pre>
    """
    
    # Basic conversion; ATX style writes "# Title" headings (the default underlines h1/h2 instead)
    markdown_output = markdownify.markdownify(html_content, heading_style=markdownify.ATX)
    
    print(f"HTML Input Length: {len(html_content)} characters")
    print(f"Markdown Output Length: {len(markdown_output)} characters")
    print(f"Conversion successful: {markdown_output is not None}")
    
    # Display first few lines of output
    markdown_lines = markdown_output.strip().split('\n')
    print(f"\nFirst 10 lines of Markdown output:")
    for i, line in enumerate(markdown_lines[:10], 1):
        print(f"  {i:2d}: {line}")
    
    # Verify key conversions
    conversions_verified = {
        'Headers': '# Main Title' in markdown_output,
        'Bold text': '**bold text**' in markdown_output,
        'Italic text': '*italic text*' in markdown_output,
        'Links': '[link to example](https://example.com)' in markdown_output,
        'List items': '* First list item' in markdown_output or '- First list item' in markdown_output,
        'Code blocks': '```\ndef hello_world():' in markdown_output,
        'Blockquotes': '> This is a blockquote with some important information.' in markdown_output
    }
    
    print(f"\nConversion Verification:")
    all_verified = True
    for element, verified in conversions_verified.items():
        status = "✓" if verified else "✗"
        print(f"  {status} {element}: {'Converted' if verified else 'Not found'}")
        all_verified &= verified
    
    print(f"\nOverall conversion success: {all_verified}")
    assert all_verified, conversions_verified
    
    # Test with strip parameter
    markdown_stripped = markdownify.markdownify(html_content, strip=['script', 'style'])
    print(f"Stripped conversion length: {len(markdown_stripped)} characters")
    
    # Test whitespace handling
    markdown_no_spaces = markdownify.markdownify(html_content.replace('\n', '').replace('    ', ' '))
    whitespace_handled = '**bold text**' in markdown_no_spaces and '[link to example](https://example.com)' in markdown_no_spaces
    assert whitespace_handled, markdown_no_spaces
    print(f"Whitespace handling: {'Success' if whitespace_handled else 'Failed'}")
    
    print("markdownify example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"markdownify example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Custom conversion with MarkdownConverter class

In [ ]:
from markdownify import ATX, MarkdownConverter

# Example 2: Custom conversion with MarkdownConverter class
print("markdownify custom MarkdownConverter")

try:
    print("Example 2 - Custom conversion with MarkdownConverter class")
    
    # Complex HTML with tables and other elements
    complex_html = """
    <div class="content">
        <h1>Document Title</h1>
        <p class="intro">Introduction paragraph with <span style="color: red;">styled text</span>.</p>
        <table border="1">
            <thead>
                <tr>
                    <th>Name</th>
                    <th>Age</th>
                    <th>City</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td>John</td>
                    <td>30</td>
                    <td>New York</td>
                </tr>
                <tr>
                    <td>Jane</td>
                    <td>25</td>
                    <td>Boston</td>
                </tr>
            </tbody>
        </table>
        <div class="note">
            <p>This is a note in a div element.</p>
        </div>
        <img src="image.jpg" alt="Sample image" width="300">
        <a href="#section1">Internal link</a>
    </div>
    """
    
    # Create custom converter with specific options
    class CustomConverter(MarkdownConverter):
        # converter methods receive parent_tags (a set of enclosing tag names)
        def convert_div(self, el, text, parent_tags):
            # Custom div handling - treat as paragraph
            if text.strip():
                return f"\n{text.strip()}\n"
            return ""
    
    # Test different conversion options
    print("Testing conversion options:")
    
    # Default converter
    default_converter = MarkdownConverter()
    default_result = default_converter.convert(complex_html)
    print(f"  Default conversion: {len(default_result)} characters")
    
    # Strip certain tags
    strip_converter = MarkdownConverter(strip=['div', 'span'])
    strip_result = strip_converter.convert(complex_html)
    print(f"  With tag stripping: {len(strip_result)} characters")
    
    # Convert certain tags
    convert_converter = MarkdownConverter(convert=['div', 'span'])
    convert_result = convert_converter.convert(complex_html)  
    print(f"  With tag conversion: {len(convert_result)} characters")
    
    # Custom converter
    custom_converter = CustomConverter()
    custom_result = custom_converter.convert(complex_html)
    print(f"  Custom converter: {len(custom_result)} characters")
    
    # ATX headings
    atx_result = MarkdownConverter(heading_style=ATX).convert(complex_html)
    
    # Analyze conversion results
    print(f"\nConversion Analysis:")
    verifications = {
        'Table': '| Name | Age | City |\n| --- | --- | --- |\n| John | 30 | New York |' in default_result,
        'Image': '![Sample image](image.jpg)' in default_result,
        'Link': '[Internal link](#section1)' in default_result,
        'Heading (default, underlined)': 'Document Title\n==============' in default_result,
        'Heading (ATX)': '# Document Title' in atx_result,
        # strip= removes the tags but keeps their text; convert= converts only the listed tags
        'strip keeps the text': 'Introduction paragraph with styled text.' in strip_result,
        'convert= leaves other tags as plain text': '|' not in convert_result and 'Sample image' not in convert_result,
        'Custom div handler': 'This is a note in a div element.\n![Sample image](image.jpg)' in custom_result,
    }
    for check, passed in verifications.items():
        print(f"  {'✓' if passed else '✗'} {check}")
    assert all(verifications.values()), verifications
    
    # Show sample output
    print(f"\nSample default conversion output (first 200 chars):")
    sample_output = default_result[:200].replace('\n', '\\n')
    print(f"  '{sample_output}'")
    
    # Test converter customization
    print(f"\nConverter Customization:")
    converter_attrs = dir(MarkdownConverter())
    method_count = len([attr for attr in converter_attrs if attr.startswith('convert_')])
    print(f"  Available convert methods: {method_count}")
    print(f"  Converter is customizable: {hasattr(MarkdownConverter, '__init__')}")
    
    # Performance comparison
    import time
    
    start_time = time.time()
    for _ in range(100):
        markdownify.markdownify(complex_html)
    function_time = time.time() - start_time
    
    start_time = time.time()
    converter = MarkdownConverter()
    for _ in range(100):
        converter.convert(complex_html)
    class_time = time.time() - start_time
    
    print(f"  Function approach (100 iterations): {function_time:.4f}s")
    print(f"  Class approach (100 iterations): {class_time:.4f}s")
    print(f"  Class is faster: {class_time < function_time}")
    
    print("markdownify example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"markdownify example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Advanced HTML processing and tag handling

In [ ]:
import markdownify
from markdownify import MarkdownConverter

# Example 3: Advanced HTML processing and tag handling
print("markdownify advanced HTML processing")

try:
    print("Example 3 - Advanced HTML processing and tag handling")
    
    # Complex HTML document with various edge cases
    advanced_html = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Sample Document</title>
        <style>
            .highlight { background: yellow; }
            .code { font-family: monospace; }
        </style>
        <script>
            function showAlert() {
                alert('Hello World!');
            }
        </script>
    </head>
    <body>
        <article>
            <header>
                <h1 id="main-title">Advanced HTML Document</h1>
                <time datetime="2023-12-01">December 1, 2023</time>
            </header>
            
            <section>
                <h2>Formatting Examples</h2>
                <p>Text with <mark class="highlight">highlighted</mark> content and 
                <abbr title="HyperText Markup Language">HTML</abbr> abbreviation.</p>
                
                <p>Math expression: <span class="math">x<sup>2</sup> + y<sub>1</sub> = z</span></p>
                
                <details>
                    <summary>Click to expand details</summary>
                    <p>Hidden content that can be revealed.</p>
                </details>
            </section>
            
            <section>
                <h2>Lists and Definitions</h2>
                <ol start="5">
                    <li>Fifth item</li>
                    <li>Sixth item with <del>strikethrough</del> and <ins>inserted</ins> text</li>
                    <li>Seventh item</li>
                </ol>
                
                <dl>
                    <dt>HTML</dt>
                    <dd>HyperText Markup Language</dd>
                    <dt>CSS</dt>
                    <dd>Cascading Style Sheets</dd>
                </dl>
            </section>
            
            <aside>
                <h3>Sidebar Information</h3>
                <p>This is supplementary information in an aside element.</p>
            </aside>
            
            <footer>
                <p>&copy; 2023 Example Company. All rights reserved.</p>
                <address>
                    Contact us at <a href="mailto:info@example.com">info@example.com</a>
                </address>
            </footer>
        </article>
    </body>
    </html>
    """
    
    # Test various stripping and conversion options
    print("Testing advanced conversion options:")
    
    # The default converter drops <script> and <style> contents
    clean_converter = MarkdownConverter()
    clean_result = clean_converter.convert(advanced_html)
    print(f"  Clean conversion (no scripts/styles): {len(clean_result)} characters")
    assert 'showAlert' not in clean_result and '.highlight' not in clean_result, clean_result
    
    # sup_symbol / sub_symbol mark superscript and subscript text
    sup_result = MarkdownConverter(sup_symbol='^', sub_symbol='~').convert(advanced_html)
    
    # Keep semantic HTML5 elements
    semantic_converter = MarkdownConverter(
        convert=['article', 'section', 'aside', 'header', 'footer', 'time', 'mark', 'details', 'summary']
    )
    semantic_result = semantic_converter.convert(advanced_html)
    print(f"  Semantic element conversion: {len(semantic_result)} characters")
    
    # strip= removes the listed tags' markup but keeps their text
    content_only_converter = MarkdownConverter(
        strip=['aside', 'footer', 'a']
    )
    content_result = content_only_converter.convert(advanced_html)
    print(f"  Tags stripped (text kept): {len(content_result)} characters")
    assert 'Contact us at info@example.com' in content_result, content_result
    
    # Advanced element analysis
    print(f"\nAdvanced Element Analysis:")
    
    # Check for specific conversions
    conversions_check = {
        'Superscript/Subscript': 'x^2^ + y~1~ = z' in sup_result,
        'Strikethrough': '~~strikethrough~~' in clean_result,
        'Inserted text': 'and inserted text' in clean_result,
        'Abbreviation': 'HTML abbreviation.' in clean_result,
        'Definition lists': 'HTML\n:   HyperText Markup Language' in clean_result,
        'Ordered lists': '5. Fifth item\n6. Sixth item' in clean_result,
        'Email links': '[info@example.com](mailto:info@example.com)' in clean_result,
        'Copyright symbol': '© 2023 Example Company' in clean_result
    }
    
    for element, found in conversions_check.items():
        status = "✓" if found else "○"
        print(f"  {status} {element}: {'Found' if found else 'Not detected'}")
    assert all(conversions_check.values()), conversions_check
    
    # Custom converter for special handling
    class AdvancedConverter(MarkdownConverter):
        # converter methods receive parent_tags (a set of enclosing tag names)
        def convert_mark(self, el, text, parent_tags):
            # Convert <mark> to highlighted text
            return f"=={text}=="
        
        def convert_abbr(self, el, text, parent_tags):
            # Convert abbreviations with full form
            title = el.get('title')
            if title:
                return f"{text} ({title})"
            return text
        
        def convert_time(self, el, text, parent_tags):
            # Convert time elements
            datetime = el.get('datetime')
            if datetime:
                return f"{text} [{datetime}]"
            return text
    
    # Test custom converter
    advanced_converter = AdvancedConverter()
    advanced_result = advanced_converter.convert(advanced_html)
    print(f"  Custom converter result: {len(advanced_result)} characters")
    
    # Check custom conversions
    custom_features = {
        'Mark highlighting': 'Text with ==highlighted== content' in advanced_result,
        'Abbr expansion': 'HTML (HyperText' in advanced_result,
        'Time formatting': '[2023-12-01]' in advanced_result
    }
    
    print(f"\nCustom Converter Features:")
    for feature, works in custom_features.items():
        status = "✓" if works else "○"
        print(f"  {status} {feature}: {'Working' if works else 'Not detected'}")
    assert all(custom_features.values()), custom_features
    
    # HTML structure analysis
    print(f"\nHTML Structure Analysis:")
    
    # Count different element types in original
    import re
    tag_pattern = r'<(\w+)[^>]*>'
    tags = re.findall(tag_pattern, advanced_html.lower())
    tag_counts = {}
    for tag in tags:
        tag_counts[tag] = tag_counts.get(tag, 0) + 1
    
    print(f"  Original HTML tags: {len(tag_counts)} unique types")
    print(f"  Most common tags: {sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)[:5]}")
    
    # Estimate markdown complexity
    markdown_lines = clean_result.split('\n')
    non_empty_lines = [line for line in markdown_lines if line.strip()]
    header_lines = [line for line in non_empty_lines if line.startswith('#')]
    list_lines = [line for line in non_empty_lines if line.strip().startswith(('*', '-', '+')) or re.match(r'^\s*\d+\.', line)]
    
    print(f"  Markdown output analysis:")
    print(f"    Total lines: {len(markdown_lines)}")
    print(f"    Content lines: {len(non_empty_lines)}")
    print(f"    Header lines: {len(header_lines)}")
    print(f"    List lines: {len(list_lines)}")
    
    # Show snippet of advanced conversion
    print(f"\nSample advanced conversion (first 300 chars):")
    sample = advanced_result[:300].strip().replace('\n', '\\n')
    print(f"  '{sample}...'")
    
    print("markdownify example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"markdownify example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Real-world web scraping and documentation workflow

In [ ]:
import markdownify
from markdownify import MarkdownConverter
import urllib.request
from io import StringIO

# Example 4: Real-world web scraping and documentation workflow
print("markdownify web scraping and documentation workflow")

try:
    print("Example 4 - Real-world web scraping and documentation workflow")
    
    # Simulate HTML from a documentation page or blog post
    documentation_html = """
    <div class="main-content">
        <nav class="breadcrumb">
            <a href="/">Home</a> &gt; <a href="/docs">Docs</a> &gt; <a href="/docs/api">API</a>
        </nav>
        
        <main>
            <h1>API Documentation</h1>
            
            <div class="toc">
                <h2>Table of Contents</h2>
                <ul>
                    <li><a href="#introduction">Introduction</a></li>
                    <li><a href="#authentication">Authentication</a></li>
                    <li><a href="#endpoints">Endpoints</a></li>
                    <li><a href="#examples">Examples</a></li>
                </ul>
            </div>
            
            <section id="introduction">
                <h2>Introduction</h2>
                <p class="lead">Welcome to our API documentation. This guide will help you integrate with our services.</p>
                
                <div class="alert alert-info">
                    <strong>Note:</strong> This API requires authentication for all requests.
                </div>
                
                <p>Our API follows <abbr title="Representational State Transfer">REST</abbr> principles and returns JSON responses.</p>
            </section>
            
            <section id="authentication">
                <h2>Authentication</h2>
                <p>Include your API key in the request header:</p>
                
                <div class="code-block">
                    <pre><code class="language-bash">curl -H "Authorization: Bearer YOUR_API_KEY" \\
     -X GET https://api.example.com/users</code></pre>
                </div>
                
                <div class="warning">
                    <p><strong>Security Warning:</strong> Never expose your API key in client-side code.</p>
                </div>
            </section>
            
            <section id="endpoints">
                <h2>Available Endpoints</h2>
                
                <div class="endpoint">
                    <h3><code>GET /users</code></h3>
                    <p>Retrieve a list of users.</p>
                    
                    <h4>Parameters</h4>
                    <table class="params-table">
                        <thead>
                            <tr>
                                <th>Parameter</th>
                                <th>Type</th>
                                <th>Required</th>
                                <th>Description</th>
                            </tr>
                        </thead>
                        <tbody>
                            <tr>
                                <td><code>page</code></td>
                                <td>integer</td>
                                <td>No</td>
                                <td>Page number for pagination (default: 1)</td>
                            </tr>
                            <tr>
                                <td><code>limit</code></td>
                                <td>integer</td>
                                <td>No</td>
                                <td>Number of results per page (default: 20, max: 100)</td>
                            </tr>
                        </tbody>
                    </table>
                    
                    <h4>Response</h4>
                    <div class="code-block">
                        <pre><code class="language-json">{
  "users": [
    {
      "id": 1,
      "name": "John Doe",
      "email": "john@example.com",
      "created_at": "2023-01-01T00:00:00Z"
    }
  ],
  "pagination": {
    "page": 1,
    "limit": 20,
    "total": 150,
    "pages": 8
  }
}</code></pre>
                    </div>
                </div>
            </section>
            
            <section id="examples">
                <h2>Code Examples</h2>
                
                <div class="example">
                    <h3>Python Example</h3>
                    <div class="code-block">
                        <pre><code class="language-python">import requests

response = requests.get(
    'https://api.example.com/users',
    headers={'Authorization': 'Bearer YOUR_API_KEY'}
)
users = response.json()['users']
print(f"Found {len(users)} users")</code></pre>
                    </div>
                </div>
            </section>
        </main>
        
        <aside class="sidebar">
            <div class="related">
                <h3>Related</h3>
                <ul>
                    <li><a href="/docs/quickstart">Quick Start Guide</a></li>
                    <li><a href="/docs/sdk">Python SDK</a></li>
                    <li><a href="/docs/webhooks">Webhooks</a></li>
                </ul>
            </div>
        </aside>
        
        <footer class="page-footer">
            <p>Last updated: <time datetime="2023-12-01T15:30:00Z">December 1, 2023 at 3:30 PM</time></p>
            <p><a href="/support">Contact Support</a> | <a href="/changelog">Changelog</a></p>
        </footer>
    </div>
    """
    
    # Clean converter for web scraping: navigation, sidebars and footers are dropped, markup and text
    # (strip= would only remove the tags themselves and keep their text)
    class WebScrapingConverter(MarkdownConverter):
        def convert_nav(self, el, text, parent_tags):
            return ''
        convert_aside = convert_footer = convert_nav
    
    # Documentation-focused converter: also turns alert and warning boxes into blockquotes
    class DocumentationConverter(WebScrapingConverter):
        def convert_div(self, el, text, *args, **kwargs):
            # Handle special div classes with flexible parameter handling
            class_name = el.get('class', [''])[0] if el.get('class') else ''
            
            if 'alert' in class_name or 'warning' in class_name:
                # Convert alerts to blockquotes
                return f"\n> {text.strip()}\n"
            elif 'code-block' in class_name:
                # Code blocks are handled by pre/code tags
                return text
            elif 'toc' in class_name:
                # Keep table of contents
                return f"\n{text}\n"
            else:
                return text
    
    # Test different conversion approaches
    print("Testing documentation conversion approaches:")
    
    # Standard conversion
    standard_converter = MarkdownConverter()
    standard_result = standard_converter.convert(documentation_html)
    print(f"  Standard conversion: {len(standard_result)} characters")
    
    # Documentation-focused conversion
    doc_converter = DocumentationConverter()
    doc_result = doc_converter.convert(documentation_html)
    print(f"  Documentation-focused: {len(doc_result)} characters")
    
    # Web scraping conversion
    scraping_converter = WebScrapingConverter()
    scraping_result = scraping_converter.convert(documentation_html)
    print(f"  Web scraping focused: {len(scraping_result)} characters")
    
    # strip= removes these tags' markup and keeps their text
    clean_converter = MarkdownConverter(strip=['nav', 'aside', 'footer', 'div'])
    clean_result = clean_converter.convert(documentation_html)
    print(f"  Tags stripped (text kept): {len(clean_result)} characters")
    
    # Analyze conversion quality
    print(f"\nConversion Quality Analysis:")
    
    # Check for preserved elements
    quality_checks = {
        'Headers preserved': any(result.count('#') > 5 for result in [standard_result, doc_result, scraping_result]),
        'Code blocks preserved': any('```' in result or '    ' in result for result in [standard_result, doc_result, scraping_result]),
        'Tables preserved': any('|' in result for result in [standard_result, doc_result, scraping_result]),
        'Links preserved': any('[' in result and '](' in result for result in [standard_result, doc_result, scraping_result]),
        'Navigation dropped': '[Home](/)' in standard_result and '[Home](/)' not in scraping_result,
        'Sidebar dropped': 'Quick Start Guide' in standard_result and 'Quick Start Guide' not in doc_result,
        'Alerts converted': '> **Note:** This API requires authentication for all requests.' in doc_result,
        'API endpoints preserved': 'GET /users' in doc_result or 'GET /users' in standard_result,
        'Code examples kept': 'import requests' in doc_result or 'import requests' in standard_result
    }
    
    for check, passed in quality_checks.items():
        status = "✓" if passed else "○"
        print(f"  {status} {check}: {'Yes' if passed else 'No'}")
    assert all(quality_checks.values()), quality_checks
    
    # Workflow simulation
    print(f"\nDocumentation Workflow Simulation:")
    
    # Step 1: Extract main content
    main_content = doc_result
    
    # Step 2: Clean up formatting
    lines = main_content.split('\n')
    cleaned_lines = []
    for line in lines:
        # Remove excessive blank lines
        if line.strip() or (cleaned_lines and cleaned_lines[-1].strip()):
            cleaned_lines.append(line)
    
    cleaned_content = '\n'.join(cleaned_lines)
    
    print(f"  Step 1 - Content extraction: {len(main_content)} chars")
    print(f"  Step 2 - Formatting cleanup: {len(cleaned_content)} chars")
    
    # Step 3: Generate table of contents
    headers = []
    for line in cleaned_lines:
        if line.startswith('#'):
            level = len(line) - len(line.lstrip('#'))
            header_text = line.lstrip('#').strip()
            if header_text:
                headers.append((level, header_text))
    
    print(f"  Step 3 - TOC generation: Found {len(headers)} headers")
    
    # Step 4: Generate final document
    toc_lines = []
    for level, header in headers:
        indent = '  ' * (level - 1)
        anchor = header.lower().replace(' ', '-').replace('/', '')
        toc_lines.append(f"{indent}- [{header}](#{anchor})")
    
    final_doc = f"# Table of Contents\n\n" + '\n'.join(toc_lines) + f"\n\n{cleaned_content}"
    
    print(f"  Step 4 - Final document: {len(final_doc)} chars")
    
    # Performance comparison
    import time
    
    test_iterations = 50
    
    print(f"\nPerformance Comparison ({test_iterations} iterations):")
    
    # Standard markdownify function
    start_time = time.time()
    for _ in range(test_iterations):
        markdownify.markdownify(documentation_html)
    function_time = time.time() - start_time
    
    # Custom documentation converter
    start_time = time.time()
    doc_converter = DocumentationConverter()
    for _ in range(test_iterations):
        doc_converter.convert(documentation_html)
    custom_time = time.time() - start_time
    
    print(f"  Standard function: {function_time:.4f}s")
    print(f"  Custom converter: {custom_time:.4f}s")
    print(f"  Performance ratio: {custom_time/function_time:.2f}x")
    
    # Show sample output
    print(f"\nSample Documentation Output (first 400 chars):")
    sample = doc_result[:400].replace('\n', '\\n')
    print(f"  '{sample}...'")
    
    # File size comparison
    print(f"\nFile Size Analysis:")
    html_size = len(documentation_html)
    markdown_size = len(doc_result)
    compression_ratio = (html_size - markdown_size) / html_size * 100
    
    print(f"  Original HTML: {html_size:,} bytes")
    print(f"  Converted Markdown: {markdown_size:,} bytes") 
    print(f"  Size reduction: {compression_ratio:.1f}%")
    
    print("markdownify example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"markdownify example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: markdownify advanced features and production workflow

In [ ]:
import markdownify
from markdownify import MarkdownConverter

# Example 5: markdownify advanced features and production workflow
print("markdownify advanced features and production workflow")

try:
    print("Example 5 - Advanced markdownify features and production workflow")
    
    # Demonstrate package structure and capabilities
    print(f"markdownify Package Overview:")
    print(f"  Module type: {type(markdownify).__name__}")
    print(f"  Package location: {markdownify.__file__ if hasattr(markdownify, '__file__') else 'Built-in'}")
    
    # Key functions and classes
    key_components = {
        'markdownify': 'Main conversion function',
        'MarkdownConverter': 'Customizable converter class'
    }
    
    print(f"  Key Components:")
    for component, description in key_components.items():
        try:
            if component == 'markdownify':
                func = getattr(markdownify, component)
                print(f"    ✓ {component}(): {description}")
            else:
                cls = getattr(markdownify, component)
                print(f"    ✓ {component}: {description}")
        except AttributeError:
            print(f"    ✗ {component}: Not available")
    
    # Production workflow demonstration
    print(f"\nProduction Workflow Pattern:")
    
    # Step 1: Input validation and preprocessing
    test_html_cases = [
        "<h1>Simple header</h1>",
        "<p>Paragraph with <strong>bold</strong> text</p>",
        "<div><p>Nested content</p></div>",
        "<table><tr><th>Header</th></tr><tr><td>Data</td></tr></table>",
        "<script>alert('test')</script><p>Clean content</p>",
        "",  # Empty input
        "<invalid>Malformed HTML</invalid>",
        "Plain text without HTML tags"
    ]
    
    print(f"  1. Input Validation:")
    valid_inputs = 0
    for i, html in enumerate(test_html_cases):
        result = markdownify.markdownify(html)
        is_valid = isinstance(result, str) and len(result) >= 0
        print(f"     ✓ Case {i+1}: {'Valid' if is_valid else 'Invalid'} - {len(html)} chars")
        if is_valid:
            valid_inputs += 1
    
    print(f"     Summary: {valid_inputs}/{len(test_html_cases)} inputs processed successfully")
    assert valid_inputs == len(test_html_cases), valid_inputs
    
    # Step 2: Configuration management
    print(f"  2. Configuration Management:")
    
    # Test different configuration options separately
    config_tests = [
        ({'strip': ['script', 'style']}, 'Strip scripts and styles'),
        ({'convert': ['div', 'span']}, 'Convert divs and spans'),
        ({}, 'Default configuration')
    ]
    
    test_html = "<div><script>alert('test')</script><p>Content</p><nav>Menu</nav></div>"
    
    expected_outputs = {
        'Strip scripts and styles': "alert('test')\n\nContent\n\nMenu",
        'Convert divs and spans': "\n\nalert('test')ContentMenu\n\n",
        'Default configuration': 'Content\n\nMenu',
    }
    for config, description in config_tests:
        converter = MarkdownConverter(**config)
        result = converter.convert(test_html)
        assert result == expected_outputs[description], (description, result)
        print(f"     ✓ {description}: {len(result)} chars")
    
    # Step 3: Performance optimization
    print(f"  3. Performance Optimization:")
    
    # Generate test content of varying sizes
    test_sizes = [100, 1000, 10000, 50000]  # Character counts
    
    for size in test_sizes:
        # Generate HTML content
        html_parts = []
        current_size = 0
        while current_size < size:
            part = f"<p>Test paragraph {current_size // 50} with some <strong>formatted</strong> content.</p>\n"
            html_parts.append(part)
            current_size += len(part)
        
        test_html = ''.join(html_parts)
        
        # Measure conversion time
        import time
        start_time = time.time()
        result = markdownify.markdownify(test_html)
        conversion_time = time.time() - start_time
        
        chars_per_second = len(test_html) / conversion_time if conversion_time > 0 else float('inf')
        
        print(f"     Size {size:>6} chars: {conversion_time:.4f}s ({chars_per_second:,.0f} chars/sec)")
    
    # Step 4: Error handling and edge cases  
    print(f"  4. Error Handling:")
    
    edge_cases = [
        ("<p>Normal content</p>", "Normal HTML"),
        ("", "Empty string"),
        ("   ", "Whitespace only"),
        ("<p><p><p>Deeply nested</p></p></p>", "Deep nesting"),
        ("<<invalid>>content<</invalid>>", "Malformed tags"),
        ("Content with no tags", "Plain text"),
        ("<p>Unicode: éñ中文🚀</p>", "Unicode content"),
        ("<p>" + "Very long content " * 1000 + "</p>", "Large content block")
    ]
    
    successful_cases = 0
    
    for html, description in edge_cases:
        result = markdownify.markdownify(html)
        status = "✓" if isinstance(result, str) else "⚠"
        print(f"     {status} {description}: {'Success' if isinstance(result, str) else 'Warning'}")
        if isinstance(result, str):
            successful_cases += 1
    
    print(f"     Edge case handling: {successful_cases}/{len(edge_cases)} successful")
    assert successful_cases == len(edge_cases), successful_cases
    
    # Step 5: Advanced customization
    print(f"  5. Advanced Customization:")
    
    # Custom converter with specialized handling
    class ProductionConverter(MarkdownConverter):
        def __init__(self):
            super().__init__(
                strip=['script', 'style', 'nav', 'footer', 'aside']
            )
        
        def convert_div(self, el, text, *args, **kwargs):
            # Custom div handling based on class with flexible parameter handling
            classes = el.get('class', [])
            if 'code-block' in classes:
                return f"\n```\n{text.strip()}\n```\n"
            elif 'warning' in classes:
                return f"\n> ⚠️ {text.strip()}\n"
            elif 'info' in classes:
                return f"\n> ℹ️ {text.strip()}\n"
            return text
    
    # Test custom converter
    custom_html = '''
    <div class="info">This is an info box</div>
    <div class="warning">This is a warning</div>
    <div class="code-block">print("Hello, World!")</div>
    <div>Regular div content</div>
    '''
    
    standard_result = markdownify.markdownify(custom_html)
    custom_converter = ProductionConverter()
    custom_result = custom_converter.convert(custom_html)
    
    print(f"     Standard conversion: {len(standard_result)} chars")
    print(f"     Custom conversion: {len(custom_result)} chars")
    print(f"     Custom features working: {'⚠️' in custom_result or 'ℹ️' in custom_result}")
    
    # Step 6: Integration patterns
    print(f"  6. Integration Patterns:")
    
    integration_patterns = [
        "Web scraping and content extraction",
        "CMS and blog content conversion",
        "Documentation generation from HTML",
        "Email template conversion",
        "Static site generation workflows",
        "API response formatting",
        "Content migration between systems",
        "Markdown-based documentation pipelines"
    ]
    
    for pattern in integration_patterns:
        print(f"     • {pattern}")
    
    # Best practices
    print(f"\nProduction Best Practices:")
    best_practices = [
        "Always validate and sanitize input HTML",
        "Use custom converters for consistent output formatting",
        "Strip unnecessary tags (scripts, styles) for security",
        "Test with various HTML structures and edge cases", 
        "Monitor performance with large content blocks",
        "Implement proper error handling and logging",
        "Cache conversion results for repeated content",
        "Consider output format compatibility requirements"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"     {i}. {practice}")
    
    # Package ecosystem integration
    print(f"\nEcosystem Integration:")
    integrations = [
        "BeautifulSoup: HTML parsing and preprocessing",
        "Requests: Web scraping and content fetching",
        "Flask/Django: Web application content processing", 
        "Pandoc: Document format conversion pipeline",
        "Mistune/CommonMark: Markdown parsing and validation",
        "PyPDF2: PDF to HTML to Markdown conversion"
    ]
    
    for integration in integrations:
        print(f"  • {integration}")
    
    # Performance characteristics
    print(f"\nPerformance Characteristics:")
    print(f"  • Conversion speed: Variable based on content size")
    print(f"  • Memory usage: Proportional to input HTML size")
    print(f"  • Processing overhead: Low for typical web content")
    print(f"  • Scalability: Suitable for batch processing workflows")
    
    print("markdownify example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"markdownify example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")